## 1. Initialize Project Environment
Import dependencies and locate Lab 1 FASTA files.

In [11]:
from __future__ import annotations

import logging
import shutil
from pathlib import Path
from typing import Dict

from Bio import SeqIO

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")

try:
    import Bio

    print("biopython", Bio.__version__)
except Exception as exc:
    logging.error("Biopython import failed: %s", exc)

biopython 1.85


## 2. Define Configuration Parameters
Set paths to source FASTA and output location.

In [12]:
from dataclasses import dataclass, asdict


def locate_repo_root() -> Path:
    """Find the repository root by looking for data/sample directory."""
    here = Path().resolve()
    for base in [here, *here.parents]:
        if (base / "data/sample").exists():
            return base
    raise FileNotFoundError("Could not locate repository root")


@dataclass
class FastaConfig:
    handle: str
    source_fasta: Path
    export_dir: Path = Path("artifacts")

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["source_fasta"] = str(info["source_fasta"])
        info["export_dir"] = str(info["export_dir"])
        return info


REPO_ROOT = locate_repo_root()
CONFIG = FastaConfig(
    handle="AndreiCod",
    source_fasta=REPO_ROOT / "data/sample/tp53_hs_transcript_NM_000546.6.fasta",
)

CONFIG.describe()

{'handle': 'AndreiCod',
 'source_fasta': '/home/rbals/git/daha-bdhb/BDHB-lab/data/sample/tp53_hs_transcript_NM_000546.6.fasta',
 'export_dir': 'artifacts'}

## 3. Implement Core Functionality
Load the FASTA file, display sequence information, and prepare for export.

In [13]:
def load_fasta(fasta_path: Path) -> list:
    """Load all records from a FASTA file."""
    if not fasta_path.exists():
        raise FileNotFoundError(f"FASTA file not found: {fasta_path}")
    records = list(SeqIO.parse(fasta_path, "fasta"))
    logging.info("Loaded %d sequence(s) from %s", len(records), fasta_path.name)
    return records


def describe_sequences(records: list) -> None:
    """Print summary information about loaded sequences."""
    for rec in records:
        gc_content = (rec.seq.count("G") + rec.seq.count("C")) / len(rec.seq) * 100
        print(f"ID: {rec.id}")
        print(f"Description: {rec.description}")
        print(f"Length: {len(rec.seq)} bp")
        print(f"GC Content: {gc_content:.2f}%")
        print(f"First 60 bp: {rec.seq[:60]}...")
        print()


records = load_fasta(CONFIG.source_fasta)
describe_sequences(records)

[INFO] Loaded 1 sequence(s) from tp53_hs_transcript_NM_000546.6.fasta


ID: NM_000546.6
Description: NM_000546.6 Homo sapiens tumor protein p53 (TP53), transcript variant 1, mRNA
Length: 2512 bp
GC Content: 53.38%
First 60 bp: CTCAAAAGTCTAGAGCCACCGTCCAGGGAGCAGGTAGCTGCTGGGCTCCGGGGACACTTT...



## 4. Validate with Unit Tests
Ensure the FASTA loading works correctly.

In [14]:
def test_load_fasta():
    recs = load_fasta(CONFIG.source_fasta)
    assert len(recs) >= 1
    assert len(recs[0].seq) > 0


def test_sequence_content():
    """Verify the sequence contains expected nucleotides."""
    valid_bases = set("ATCGN")
    for rec in records:
        seq_bases = set(str(rec.seq).upper())
        assert seq_bases.issubset(valid_bases), f"Invalid bases in {rec.id}"


test_load_fasta()
test_sequence_content()
print("All inline tests passed.")

[INFO] Loaded 1 sequence(s) from tp53_hs_transcript_NM_000546.6.fasta


All inline tests passed.


## 5. Export Results
Copy the FASTA file to the artifacts directory with the assignment naming convention.

In [15]:
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

out_file = EXPORT_DIR / "task2_tp53_sequence.fasta"

# Copy the FASTA file
shutil.copy(CONFIG.source_fasta, out_file)

# Verify the copy
copied_records = list(SeqIO.parse(out_file, "fasta"))
assert len(copied_records) == len(records)
assert str(copied_records[0].seq) == str(records[0].seq)

print(f"[OK] FASTA file saved to: {out_file.resolve()}")
print(f"     Contains {len(copied_records)} sequence(s)")

[OK] FASTA file saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/03_formats&NGS/assignments/artifacts/task2_tp53_sequence.fasta
     Contains 1 sequence(s)
